In [74]:
#1 — Imports
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [75]:
#2 — Load Environment
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

assert GEMINI_API_KEY
assert PINECONE_API_KEY

In [76]:
#3 — Configuration Class
@dataclass(slots=True)
class RetrieverConfig:
    index_name:str
    embedding_model:str
    embedding_dimension:int
    top_k:int=5


In [77]:
config = RetrieverConfig(
    index_name="supportsphere-ai",
    embedding_model="models/gemini-embedding-001",
    embedding_dimension=768,
    top_k=5,
)

In [78]:
#4 — Build Retriever
class PineconeRetriever:

    def __init__(self, config: RetrieverConfig):
        self.config = config
        self.pc = Pinecone(api_key=PINECONE_API_KEY)
        self.index = self.pc.Index(config.index_name)
        self.embedding_model = (GoogleGenerativeAIEmbeddings(
                                model=config.embedding_model,
                                output_dimensionality=config.embedding_dimension))
        
    
    #Retrieve Method
    def retrieve(self, question: str, company: Optional[str] = None, 
             product_area: Optional[str] = None,top_k: Optional[int] = None,):

        query_vector = self.embedding_model.embed_query(question)
        filters = {}
        if company:
            filters["company"] = company.lower()

        if product_area:
            filters["product_area"] = product_area

        results = self.index.query(vector=query_vector,
                                    top_k=top_k or self.config.top_k,
                                    include_metadata=True,
                                    filter=filters if filters else None,)

        return results.matches
    
    #Build Context
    @staticmethod
    def build_context(matches):
        return "\n\n".join(match["metadata"]["text"] for match in matches)
    
    #Pretty Printer
    @staticmethod
    def print_matches(matches):

        for i, match in enumerate(matches, start=1):
            print("="*80)
            print(f"Rank : {i}")
            print(f"Score : {match['score']:.4f}")
            print()
            print("Company :", match["metadata"]["company"])
            print("Product :", match["metadata"]["product_area"])
            print("Title :", match["metadata"]["title"])
            print()
            print(match["metadata"]["text"][:500])


In [79]:
#5 — Initialize
retriever = PineconeRetriever(config)

In [80]:
#6 — First Query
matches = retriever.retrieve(question="How do I reset my Claude password?")

retriever.print_matches(matches)

Rank : 1
Score : 0.7820

Company : claude
Product : claude
Title : "Logging in to your Claude account"

How can I set a password for my Claude account?
It’s not possible to create a dedicated password for your Claude account at this time.
Can I use my Claude account across multiple devices?
To use your Claude account across multiple devices, enter the same email address you use to log in on your usual device. For example, if you signed up for your Claude account on a web browser, you can access the same account on the Claude mobile apps by entering the same email address when you log in.
I have mu
Rank : 2
Score : 0.7722

Company : claude
Product : claude
Title : "Logging in to your Claude account"

**Note: **Business, government, or .edu email users may need to contact their IT department to adjust email security settings.
I received the email but I'm still having trouble logging in.
If you received the login email but can’t log in with the link or code, take note of the error message

In [81]:
#7 — Company Filter
matches = retriever.retrieve(

    question="Reset password",

    company="hackerrank",

)

retriever.print_matches(matches)

Rank : 1
Score : 0.7056

Company : hackerrank
Product : settings
Title : "Update or Reset Password Updating password Resetting password"

Resetting password
To reset a forgotten password:
Go to the HackerRank for Work login page.
Click **Forgot Password?** under the password field.
 <img src="https://assets.usepylon.com/e6a58e21-be80-4777-9eaf-f73beeee94d9%2Fba2098a0-25fd-4540-be8b-22162546efef-AD_4nXcfZaFqveIjoF0r78h6u7fu_b1gf2jWGjsUpZbCwCr6rdvYuke6rpzZhzI1LPo2LBGmjEw1KnssNTIdqkhvoOyjKrE2mZbRfm9y4GLDdSQ7BfFhOhD7Kjxu_4Is6fGgzGRrcN6IjQ-4d4309e9-1afe-454b-930a-df09793c8234?Expires=253370764800&amp;Signature=VRYahrGPT-g55WR8P1-HyUiYY
Rank : 2
Score : 0.7032

Company : hackerrank
Product : hackerrank_community
Title : "Update or Reset Password"

Resetting password
To reset a forgotten password:
Go to the HackerRank Community login page.
Click **Forgot Password?** under the password field.
Enter the email address or username associated with your account and select the verification checkbox.

In [82]:
#8 — Product Area Filter
matches = retriever.retrieve(

    question="How do I access Claude?",

    company="claude",

    product_area="amazon_bedrock",
)
retriever.print_matches(matches)

Rank : 1
Score : 0.7814

Company : claude
Product : amazon_bedrock
Title : "How do I get access to Claude in Amazon Bedrock?"

How do I get access to Claude in Amazon Bedrock?
_Last updated: 2026-03-16T21:16:50Z_
Get started with Claude in Amazon Bedrock by visiting the Amazon Bedrock console. For a step-by-step walkthrough on how to request Claude model access in the Amazon Bedrock console, view this blog.
Related Articles
What is Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
Where do I find Claude in Amazon Bedrock documentation?
What AWS Regions are Claude models ava
Rank : 2
Score : 0.7401

Company : claude
Product : amazon_bedrock
Title : "What is Amazon Bedrock?"

You can learn more about Anthropic’s Claude models in Amazon Bedrock here.
Related Articles
How do I get access to Claude in Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
What AWS Regions are Claude models available in 

In [83]:
#9 — Build Context
context = retriever.build_context(matches)

print(context[:3000])

How do I get access to Claude in Amazon Bedrock?
_Last updated: 2026-03-16T21:16:50Z_
Get started with Claude in Amazon Bedrock by visiting the Amazon Bedrock console. For a step-by-step walkthrough on how to request Claude model access in the Amazon Bedrock console, view this blog.
Related Articles
What is Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
Where do I find Claude in Amazon Bedrock documentation?
What AWS Regions are Claude models available in Amazon Bedrock?
Claude Code FAQ

You can learn more about Anthropic’s Claude models in Amazon Bedrock here.
Related Articles
How do I get access to Claude in Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
What AWS Regions are Claude models available in Amazon Bedrock?
Public Sector FAQs
Use Claude for Excel, PowerPoint, and Word with third-party platforms

Related Articles
What is Amazon Bedrock?
What AWS Regions are Claude models avail

In [84]:
#10 — Notebook Summary
print("=" * 60)

print("PRODUCTION RETRIEVER COMPLETE")

print("=" * 60)

print("Retriever Class ✓")

print("Semantic Search ✓")

print("Metadata Filtering ✓")

print("Context Builder ✓")

PRODUCTION RETRIEVER COMPLETE
Retriever Class ✓
Semantic Search ✓
Metadata Filtering ✓
Context Builder ✓


                    LangGraph
                         │
                         ▼
                BaseRetriever (ABC)
                         │
        ┌────────────────┴────────────────┐
        │                                 │
        ▼                                 ▼
FAISSRetriever                 PineconeRetriever
        │                                 │
        ▼                                 ▼
     FAISS                         Pinecone Cloud

     

                    BaseRetriever
                          │
      ┌──────────────────┬───────────────────────────┐
      ▼                  ▼                           ▼
FAISSRetriever      |    PineconeRetriever       |QdrantRetriever



## Project Structure (Phase 2)

In [134]:
#1 — Imports
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import os
import pickle

import faiss
import numpy as np

from dotenv import load_dotenv

from pinecone import Pinecone

from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [135]:
#2 — Environment
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [136]:
#3 — Config
@dataclass(slots=True)
class RetrieverConfig:

    embedding_model: str

    embedding_dimension: int

    top_k: int = 5

In [137]:
#4 — BaseRetriever
class BaseRetriever(ABC):

    @abstractmethod
    def retrieve(self, question: str, company: Optional[str] = None,
                product_area: Optional[str] = None, top_k: Optional[int] = None,):
        """Return retrieved matches."""
        pass

    @abstractmethod
    def build_context(self, matches,) -> str:
        """Build LLM context."""
        pass

In [138]:
#5 — BaseVectorRetriever
class BaseVectorRetriever(BaseRetriever):

    def __init__(self, config: RetrieverConfig,):
        self.config = config
        self.embedding_model = GoogleGenerativeAIEmbeddings(
                                model=config.embedding_model,
                                output_dimensionality=config.embedding_dimension,)
    

    #Shared embedding function
    def embed_query(self, question: str,):
        return self.embedding_model.embed_query(question)


    #Shared context builder
    def build_context(self, matches,):
        texts = []
        for match in matches:            
            if "text" in match:
                texts.append(match["text"])
            elif isinstance(match, dict):
                texts.append(match["metadata"]["text"])
            else:
                texts.append(match.metadata["text"])

        return "\n\n".join(texts)    


    #Shared pretty printer
    def print_matches(self, matches,):

        for i, match in enumerate(matches, start=1):
            print("=" * 80)
            print(f"Rank : {i}")

            if isinstance(match, dict):
                metadata = match["metadata"]
                score = match.get("score")

            else:
                metadata = match.metadata
                score = match.score

            print(f"Score : {score:.4f}")
            print()
            print("Company :", metadata["company"])
            print("Product :", metadata["product_area"])
            print("Title :", metadata["title"])
            print()
            print(metadata.get("text", "Text not stored in metadata"))

In [139]:
#6 — FAISSRetriever
class FAISSRetriever(BaseVectorRetriever):

    def __init__(self, config, index_path, metadata_path,):

        super().__init__(config)

        self.index = faiss.read_index(index_path)

        with open(metadata_path, "rb") as f:

            self.metadata = pickle.load(f)

    def retrieve(self, question, company=None, product_area=None, top_k=None,):

        query = self.embed_query(question)

        query = np.array([query], dtype=np.float32,)

        distances, indices = self.index.search(query, top_k or self.config.top_k,)

        matches = []

        for distance, idx in zip(distances[0],indices[0],):

            doc = self.metadata[idx]

            if company:

                if doc["metadata"]["company"] != company.lower():
                    continue

            if product_area:

                if doc["metadata"]["product_area"] != product_area:
                    continue

            matches.append({
                "score": float(distance),
                "metadata": doc["metadata"],
                "text": doc['text'],
            })

        return matches

In [140]:
#7 — PineconeRetriever
class PineconeRetriever(BaseVectorRetriever):

    def __init__(self, config, index_name,):
        super().__init__(config)
        self.pc = Pinecone(api_key=PINECONE_API_KEY)
        self.index = self.pc.Index(index_name)
    
    def retrieve(self, question, company=None, product_area=None, top_k=None,):

        query = self.embed_query(question)
        filters = {}

        if company:
            filters["company"] = company.lower()

        if product_area:
            filters["product_area"] = product_area

        response = self.index.query(vector=query,
                                    top_k=top_k or self.config.top_k,
                                    include_metadata=True,
                                    filter=filters if filters else None,)

        matches = []

        for match in response.matches:
            matches.append({"score": float(match.score), 
                            "metadata": dict(match.metadata),})

        return matches    

In [141]:
#8 — Configuration
config = RetrieverConfig(

    embedding_model="models/gemini-embedding-001",

    embedding_dimension=768,

    top_k=5,
)

In [142]:
#9 — Create FAISS Retriever
faiss_retriever = FAISSRetriever(

    config=config,

    index_path="../artifacts/faiss_index.index",

    metadata_path="../artifacts/faiss_metadata.pkl",
)

In [143]:
#10 — Create Pinecone Retriever
pinecone_retriever = PineconeRetriever(

    config=config,

    index_name="supportsphere-ai",
)

In [144]:
#11 — Test FAISS
matches = faiss_retriever.retrieve(
    question="How do I reset my password?",
)

len(matches)

5

In [145]:
matches

[{'score': 0.18754839897155762,
  'metadata': {'document_id': '99aa25e3ed5d7b3f8adf97d0d769abf0a7300794e837d73e24a16e80eade1625',
   'document_hash': '8143f2b0dc3beb3ca89af0788a0734fe330407b24b1c2374a92311e92c273e41',
   'company': 'hackerrank',
   'product_area': 'hackerrank_community',
   'title': '"Update or Reset Password"',
   'url': '"https://help.hackerrank.com/articles/2403570133-update-or-reset-password"',
   'file_name': '2403570133-update-or-reset-password.md',
   'source_path': 'hackerrank\\hackerrank_community\\account-settings\\manage-account\\2403570133-update-or-reset-password.md',
   'extension': '.md',
   'source_type': 'markdown',
   'schema_version': '1.0',
   'indexed_at': '2026-07-07T20:51:18.048084+00:00',
   'last_modified': '2026-07-02T21:53:33.932791+00:00',
   'chunk_number': 1,
   'chunk_id': '614112679a7b7890d3ea316699606f24814652c4e43929597feae470a1319d0c',
   'content_hash': 'a9148830a0057ae00f6bee9c10f9b2a095d20c01efeb67f03ac888301442c638',
   'chunk_siz

In [146]:
context = faiss_retriever.build_context(matches)

print(context[:2000])

Resetting password
To reset a forgotten password:
Go to the HackerRank Community login page.
Click **Forgot Password?** under the password field.
Enter the email address or username associated with your account and select the verification checkbox.
Click **Send Reset Link**.
Check your email for a reset link from HackerRank. If you do not receive an email, check your spam folder or verify that you used the correct email or username.
Follow the instructions in the reset email to create a new password.

Resetting password
To reset a forgotten password:
Go to the HackerRank for Work login page.
Click **Forgot Password?** under the password field.
 <img src="https://assets.usepylon.com/e6a58e21-be80-4777-9eaf-f73beeee94d9%2Fba2098a0-25fd-4540-be8b-22162546efef-AD_4nXcfZaFqveIjoF0r78h6u7fu_b1gf2jWGjsUpZbCwCr6rdvYuke6rpzZhzI1LPo2LBGmjEw1KnssNTIdqkhvoOyjKrE2mZbRfm9y4GLDdSQ7BfFhOhD7Kjxu_4Is6fGgzGRrcN6IjQ-4d4309e9-1afe-454b-930a-df09793c8234?Expires=253370764800&amp;Signature=VRYahrGPT-g55WR8P1

In [49]:
for m in matches:
    print(m["metadata"]["company"])

hackerrank
hackerrank
hackerrank
hackerrank
hackerrank


In [147]:
#12 — Test Pinecone
matches = pinecone_retriever.retrieve(

    question="How do I reset my password?",

    company="claude",
)

pinecone_retriever.print_matches(matches)

Rank : 1
Score : 0.6448

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

If you requested the login email and clicked the link using the same device, you'll be automatically redirected to your logged-in Console account.
Clicking the link on another device
If you requested the login email and clicked the link using a different device (requesting from a web browser and clicking the email link on a phone, for example), then you will still see a "Sign in to Claude Console" button in the email, but clicking it will generate a verification code. You should enter this code on the original device where you requested the login email to authenticate.
Troubleshooting
I entered my email address but I haven't received my login email.
If you requested a login email but you haven't received it yet, do the following:
Check your spam and junk folders for emails from @mail.anthropic.com.
Check your server's quarantined emails.
Whitelist @mail.anthropic.com

In [149]:
#13 — Build Context
context = pinecone_retriever.build_context(matches)

print(context[:2000])

#context = faiss_retriever.build_context(matches)

#print(context[:2000])

If you requested the login email and clicked the link using the same device, you'll be automatically redirected to your logged-in Console account.
Clicking the link on another device
If you requested the login email and clicked the link using a different device (requesting from a web browser and clicking the email link on a phone, for example), then you will still see a "Sign in to Claude Console" button in the email, but clicking it will generate a verification code. You should enter this code on the original device where you requested the login email to authenticate.
Troubleshooting
I entered my email address but I haven't received my login email.
If you requested a login email but you haven't received it yet, do the following:
Check your spam and junk folders for emails from @mail.anthropic.com.
Check your server's quarantined emails.
Whitelist @mail.anthropic.com in your email settings.
Try logging in again after a few minutes.

Check your server's quarantined emails.
Whitelist @ma

## HybridFAISSRetriever

In [69]:
class HybridFAISSRetriever(BaseVectorRetriever):

    def __init__(self, config, artifacts_path,):

        super().__init__(config)

        artifacts_path = Path(artifacts_path)

        self.indexes = {

            "global":
                faiss.read_index(
                    str(
                        artifacts_path /
                        "global.index"
                    )
                ),

            "claude":
                faiss.read_index(
                    str(
                        artifacts_path /
                        "claude.index"
                    )
                ),

            "visa":
                faiss.read_index(
                    str(
                        artifacts_path /
                        "visa.index"
                    )
                ),

            "hackerrank":
                faiss.read_index(
                    str(
                        artifacts_path /
                        "hackerrank.index"
                    )
                ),
        }

        with open(

            artifacts_path /
            "metadata_store.pkl",

            "rb"

        ) as f:

            self.metadata = pickle.load(f)

    def retrieve(self, question, company=None, product_area=None, top_k=None,):

        query = self.embed_query(question)

        query = np.array(
            [query],
            dtype=np.float32
        )

        company = (
            company.lower()
            if company
            else "global"
        )

        index = self.indexes[
            company
        ]

        metadata = self.metadata[
            company
        ]

        distances, indices = index.search(

            query,

            top_k or self.config.top_k
        )

        matches = []

        for distance, idx in zip(

            distances[0],

            indices[0],

        ):

            doc = metadata[idx]

            if product_area:

                if (
                    doc["metadata"][
                        "product_area"
                    ]
                    != product_area
                ):

                    continue

            matches.append(

                {

                    "score": float(distance),

                    "metadata": doc["metadata"]

                }

            )

        return matches


In [70]:
#9 — Create FAISS Retriever
retriever = HybridFAISSRetriever(

    config=config,

    artifacts_path= Path("../artifacts"),
)

In [71]:
matches = retriever.retrieve( question="Reset password",company="claude")

In [72]:
matches

[{'score': 0.25210192799568176,
  'metadata': {'document_id': '467e2f8cf4bcb8c02d2dd88bdc22036004acae9e20b75a02d82b332ab0b94220',
   'document_hash': '796d5930ed4eac8a0ab732301bc2383bff357b5ca5deac7699559d35421a1b18',
   'company': 'claude',
   'product_area': 'identity_management_sso_jit_scim',
   'title': '"Switching to a different Identity Provider (IdP)"',
   'url': '"https://support.claude.com/en/articles/13443687-switching-to-a-different-identity-provider-idp"',
   'file_name': '13443687-switching-to-a-different-identity-provider-idp.md',
   'source_path': 'claude\\identity-management-sso-jit-scim\\13443687-switching-to-a-different-identity-provider-idp.md',
   'extension': '.md',
   'source_type': 'markdown',
   'schema_version': '1.0',
   'indexed_at': '2026-07-07T20:51:17.931160+00:00',
   'last_modified': '2026-07-02T21:53:33.858335+00:00',
   'chunk_number': 3,
   'chunk_id': 'e838231e6599d064ffef392b8fe58c06e70934a261423044e17c9d7ca60edda7',
   'content_hash': 'd79af0f7f890

In [73]:
retriever.print_matches(matches)

Rank : 1
Score : 0.2521

Company : claude
Product : identity_management_sso_jit_scim
Title : "Switching to a different Identity Provider (IdP)"

Text not stored in metadata
Rank : 2
Score : 0.2524

Company : claude
Product : claude
Title : "Logging in to your Claude account"

Text not stored in metadata
Rank : 3
Score : 0.2535

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

Text not stored in metadata
Rank : 4
Score : 0.2547

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

Text not stored in metadata
Rank : 5
Score : 0.2564

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

Text not stored in metadata
